# Doppler Shadow Removal Test

This notebook exercises the current Doppler-shadow implementation on synthetic time-series data.

It does three things:
1. Builds a synthetic Doppler shadow using the KELT-20b / Duck24 system parameters.
2. Injects that shadow into noisy residual spectra and runs `remove_doppler_shadow()`.
3. Verifies that the retrieval-prep helper applies the same correction path used by `prepare_retrieval_timeseries.py`.

Run this notebook inside the retrieval environment so the repo dependencies are available.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (cwd, cwd.parent)
        if (candidate / "config").exists() and (candidate / "dataio").exists()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(
        f"Could not find repo root from cwd={cwd}. Start the notebook from the repo or notebooks/ directory."
    )
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Repo root: {repo_root}")

import numpy as np
import matplotlib.pyplot as plt

from config.planets_config import PLANETS
from dataio.horus import compute_doppler_shadow, remove_doppler_shadow
from dataio.prepare_retrieval_timeseries import _apply_default_doppler_shadow

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["image.origin"] = "lower"
plt.rcParams["axes.grid"] = False


In [ ]:
planet_cfg = PLANETS["KELT-20b"]["Duck24"]

planet_params = {
    "rp_rs": float(planet_cfg["rp_rs"]),
    "b": float(planet_cfg["b"]),
    "lambda_angle": float(planet_cfg["lambda_angle"]),
    "a_rs": float(planet_cfg["a_rs"]),
    "period": float(planet_cfg["period"]),
}
stellar_params = {
    "vsini": float(planet_cfg["v_sini_star"]),
    "gamma1": float(planet_cfg["gamma1"]),
    "gamma2": float(planet_cfg["gamma2"]),
}

n_exp = 25
npix = 1800
phase = np.linspace(-0.03, 0.03, n_exp)
wave = np.linspace(6497.0, 6503.0, npix)

v_grid = (wave - np.median(wave)) / np.median(wave) * 2.9979e5
shadow_profiles, baseline = compute_doppler_shadow(
    phase=phase,
    vsini=stellar_params["vsini"],
    lambda_angle=planet_params["lambda_angle"],
    b=planet_params["b"],
    rp_rs=planet_params["rp_rs"],
    gamma1=stellar_params["gamma1"],
    gamma2=stellar_params["gamma2"],
    v_grid=v_grid,
    a_rs=planet_params["a_rs"],
    period=planet_params["period"],
)

shadow_profiles.shape, baseline.shape


In [ ]:
rng = np.random.default_rng(7)
noise_std = 0.003
injected_scale = 0.75

noise_only = rng.normal(0.0, noise_std, size=shadow_profiles.shape)
residual_flux = noise_only + injected_scale * shadow_profiles

corrected_flux, shadow_model, fit_info = remove_doppler_shadow(
    residual_flux,
    wave,
    phase,
    planet_params,
    stellar_params,
)

before_shadow_rms = np.sqrt(np.mean((residual_flux - noise_only) ** 2))
after_shadow_rms = np.sqrt(np.mean((corrected_flux - noise_only) ** 2))
noise_rms = np.sqrt(np.mean(noise_only ** 2))

print(f"Injected scaling: {injected_scale:.3f}")
print(f"Recovered scaling: {fit_info['scaling']:.3f}")
print(f"Noise RMS: {noise_rms:.5f}")
print(f"Shadow RMS before removal: {before_shadow_rms:.5f}")
print(f"Shadow RMS after removal:  {after_shadow_rms:.5f}")

assert after_shadow_rms < before_shadow_rms * 0.2, "Shadow removal did not reduce the injected signal enough."


In [ ]:
prep_corrected, prep_status = _apply_default_doppler_shadow(
    residual_flux,
    wave,
    phase,
    planet_cfg=planet_cfg,
    subtract_median=True,
)

np.testing.assert_allclose(prep_corrected, corrected_flux)
assert prep_status["applied"] is True
assert abs(prep_status["scaling"] - fit_info["scaling"]) < 1e-12

prep_status


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
extent = [wave.min(), wave.max(), phase.min(), phase.max()]

im0 = axes[0].imshow(residual_flux, aspect="auto", extent=extent, cmap="RdBu_r")
axes[0].set_title("Injected residual cube")
axes[0].set_xlabel("Wavelength [A]")
axes[0].set_ylabel("Orbital phase")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(shadow_model, aspect="auto", extent=extent, cmap="RdBu_r")
axes[1].set_title("Recovered shadow model")
axes[1].set_xlabel("Wavelength [A]")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(corrected_flux, aspect="auto", extent=extent, cmap="RdBu_r")
axes[2].set_title("After removal")
axes[2].set_xlabel("Wavelength [A]")
fig.colorbar(im2, ax=axes[2], shrink=0.8)

fig.tight_layout()


In [ ]:
mid = n_exp // 2
plt.figure(figsize=(12, 4))
plt.plot(wave, residual_flux[mid], label="Injected residual", alpha=0.8)
plt.plot(wave, shadow_model[mid], label="Recovered shadow model", alpha=0.8)
plt.plot(wave, corrected_flux[mid], label="Corrected residual", alpha=0.8)
plt.xlabel("Wavelength [A]")
plt.ylabel("Residual flux")
plt.title(f"Mid-transit exposure at phase={phase[mid]:+.4f}")
plt.legend()
plt.tight_layout()
